# RxHandBD: checks, inference, and optional GPU training
Open in VS Code/Jupyter, Colab, or Kaggle. Upload/mount this project with Dataset/RxHandBD first. Running all cells with the defaults does **not** install packages, download weights, rent GPUs, or train. Read docs/GPU_OPTIONS.md before enabling training. Do not upload real identifiable prescriptions to a cloud notebook.

In [ ]:
from pathlib import Path
import os, subprocess, sys
PROJECT = Path.cwd()  # edit for your Colab/Kaggle mount
if not (PROJECT / 'scripts/train.py').exists() and (PROJECT.parent / 'scripts/train.py').exists():
    PROJECT = PROJECT.parent
assert (PROJECT / 'Dataset/RxHandBD/train.csv').exists(), 'Set PROJECT to the uploaded project root'
os.chdir(PROJECT)
INSTALL_DEPS = False
RUN_INFERENCE = False
RUN_TRAINING = False


In [ ]:
if INSTALL_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
else:
    print('Dependency installation disabled. Use the project environment or enable explicitly.')


## Data-only preflight
This validates every image and split. No weights are loaded and no training steps run.

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.train'], check=True)


## Optional inference only
The first enabled run downloads the pinned pretrained model. Two samples are a smoke test, not an accuracy benchmark.

In [ ]:
if RUN_INFERENCE:
    subprocess.run([sys.executable, '-m', 'scripts.evaluate', '--limit', '2'], check=True)
else:
    print('Inference disabled.')


## Optional GPU fine-tuning
Only enable after choosing a GPU and editing configs/pipeline.json if needed. The test split is never used for training/model selection. Save/download checkpoints from the cloud disk before ending your session.

In [ ]:
if RUN_TRAINING:
    import torch
    assert torch.cuda.is_available(), 'Select a CUDA GPU runtime first'
    subprocess.run([sys.executable, '-m', 'scripts.train', '--execute'], check=True)
else:
    print('Training disabled: RUN_TRAINING is False.')


## Evaluate a trained model later
After training, run `python -m scripts.evaluate --model checkpoints/trocr_rxhandbd/best --split test`. Keep the test set untouched until model selection is finished. Generated outputs are not validated medicine identities.